In [ ]:
import os
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings
from urllib.parse import quote_plus
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document
from sqlalchemy import select, text
from typing import List, Optional, Dict, Any, Tuple
import logging



embeddings = OpenAIEmbeddings(
    model=os.getenv("EMBEDDING_MODEL_NAME"),
    base_url=os.getenv("EMBEDDING_BASE_URL"),
    api_key=os.getenv("EMBEDDING_API_KEY"),
)


# -----------------------------------------------------
# PostgreSQL Configuration
POSTGRESQL_DB_USER = os.getenv("POSTGRESQL_DB_USER")
POSTGRESQL_DB_PASSWORD = os.getenv("POSTGRESQL_DB_PASSWORD")
POSTGRESQL_DB_HOST = os.getenv("POSTGRESQL_DB_HOST")
POSTGRESQL_DB_PORT = os.getenv("POSTGRESQL_DB_PORT", "5432")
POSTGRESQL_DB_NAME = os.getenv("POSTGRESQL_DB_NAME")
pg_password_encoded = quote_plus(POSTGRESQL_DB_PASSWORD)

POSTGRESQL_DATABASE_URL = (
    f"postgresql+psycopg://{POSTGRESQL_DB_USER}:{pg_password_encoded}@"
    f"{POSTGRESQL_DB_HOST}:{POSTGRESQL_DB_PORT}/{POSTGRESQL_DB_NAME}"
)

In [ ]:
class HybridRAGStore(PGVector):
    """
    Hybrid RAG Store: Supports both vector search and keyword search

    Features:
    - Vector search (pgvector)
    - Keyword full-text search (pgroonga)
    - Hybrid search (RRF fusion)
    - Metadata filtering
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.logger = logging.getLogger(__name__)

    # ==================== Index Management ====================

    def create_search_indexes(self) -> None:
        """
        Create full-text search and metadata filtering indexes
        Automatically called during initialization
        """
        with self._make_sync_session() as session:
            # Create pgroonga full-text search index
            try:

                statement = text(
                    """
                    CREATE INDEX IF NOT EXISTS pgroonga_document_idx 
                    ON langchain_pg_embedding 
                    USING pgroonga (document);
                    """
                )

                session.execute(statement)
                self.logger.info("Created pgroonga full-text search index")
            except Exception as e:
                self.logger.warning(f"Failed to create pgroonga index: {e}")

            # Create GIN index for JSONB metadata filtering
            try:
                session.execute(
                    text(
                        """
                    CREATE INDEX IF NOT EXISTS ix_cmetadata_gin 
                    ON langchain_pg_embedding 
                    USING GIN (cmetadata jsonb_path_ops);
                """
                    )
                )
                self.logger.info("Created GIN index for JSONB metadata")
            except Exception as e:
                self.logger.warning(f"Failed to create GIN index: {e}")

            session.commit()

    async def acreate_search_indexes(self) -> None:
        """Async version of create_search_indexes"""
        with self._make_sync_session() as session:
            # Create pgroonga full-text search index
            try:
                session.execute(
                    text(
                        """
                    CREATE INDEX IF NOT EXISTS pgroonga_document_idx 
                    ON langchain_pg_embedding 
                    USING pgroonga (document);
                """
                    )
                )
                self.logger.info("Created pgroonga full-text search index")
            except Exception as e:
                self.logger.warning(f"Failed to create pgroonga index: {e}")

            # Create GIN index for JSONB metadata filtering
            try:
                session.execute(
                    text(
                        """
                    CREATE INDEX IF NOT EXISTS ix_cmetadata_gin 
                    ON langchain_pg_embedding 
                    USING GIN (cmetadata jsonb_path_ops);
                """
                    )
                )
                self.logger.info("Created GIN index for JSONB metadata")
            except Exception as e:
                self.logger.warning(f"Failed to create GIN index: {e}")

            session.commit()

    def create_vector_index(self, index_type: str = "hnsw") -> None:
        """
        Create vector search index (optional)

        Args:
            index_type: 'hnsw' or 'ivfflat'
        """
        with self._make_sync_session() as session:
            if index_type == "hnsw":
                try:
                    session.execute(
                        text(
                            """
                            CREATE INDEX IF NOT EXISTS embedding_hnsw_idx 
                            ON langchain_pg_embedding 
                            USING hnsw (embedding vector_cosine_ops)
                            WITH (m = 16, ef_construction = 64);
                            """
                        )
                    )
                    self.logger.info("Created HNSW vector index")
                except Exception as e:
                    self.logger.warning(f"Failed to create HNSW index: {e}")
            elif index_type == "ivfflat":
                try:
                    session.execute(
                        text(
                            """
                        CREATE INDEX IF NOT EXISTS embedding_ivfflat_idx 
                        ON langchain_pg_embedding 
                        USING ivfflat (embedding vector_cosine_ops)
                        WITH (lists = 100);
                    """
                        )
                    )
                    self.logger.info("Created IVFFlat vector index")
                except Exception as e:
                    self.logger.warning(f"Failed to create IVFFlat index: {e}")

            session.commit()

    async def acreate_vector_index(self, index_type: str = "hnsw") -> None:
        """Async version of create_vector_index"""
        with self._make_sync_session() as session:
            if index_type == "hnsw":
                try:
                    session.execute(
                        text(
                            """
                        CREATE INDEX IF NOT EXISTS embedding_hnsw_idx 
                        ON langchain_pg_embedding 
                        USING hnsw (embedding vector_cosine_ops)
                        WITH (m = 16, ef_construction = 64);
                    """
                        )
                    )
                    self.logger.info("Created HNSW vector index")
                except Exception as e:
                    self.logger.warning(f"Failed to create HNSW index: {e}")
            elif index_type == "ivfflat":
                try:
                    session.execute(
                        text(
                            """
                        CREATE INDEX IF NOT EXISTS embedding_ivfflat_idx 
                        ON langchain_pg_embedding 
                        USING ivfflat (embedding vector_cosine_ops)
                        WITH (lists = 100);
                    """
                        )
                    )
                    self.logger.info("Created IVFFlat vector index")
                except Exception as e:
                    self.logger.warning(f"Failed to create IVFFlat index: {e}")

            session.commit()

    def ensure_indexes(self) -> None:
        """Ensure all indexes exist"""
        self.create_search_indexes()
        # self.create_vector_index()  # Optional: enable based on data volume

    async def aensure_indexes(self) -> None:
        """Async version of ensure_indexes"""
        await self.acreate_search_indexes()
        # await self.acreate_vector_index()  # Optional

    # ==================== Keyword Search ====================

    def keyword_search(
        self,
        query: str,
        k: int = 10,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Document]:
        """
        Perform keyword full-text search using pgroonga
        """
        with self._make_sync_session() as session:
            collection = self.get_collection(session)
            if not collection:
                raise ValueError("Collection not found")

            filter_by = [self.EmbeddingStore.collection_id == collection.uuid]

            if filter:
                filter_clauses = self._create_filter_clause(filter)
                if filter_clauses is not None:
                    filter_by.append(filter_clauses)

            # Fix: Use string directly as pgroonga query condition
            stmt = (
                select(self.EmbeddingStore)
                .filter(*filter_by)
                .order_by(self.EmbeddingStore.document.op("@@")(query).desc())
                .limit(k)
            )

            results = session.execute(stmt).scalars().all()

            return [
                Document(
                    id=result.id,
                    page_content=result.document,
                    metadata=result.cmetadata or {},
                )
                for result in results
            ]

    async def akeyword_search(
        self,
        query: str,
        k: int = 10,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Document]:
        """Async version of keyword_search"""
        with self._make_sync_session() as session:
            collection = self.get_collection(session)
            if not collection:
                raise ValueError("Collection not found")

            filter_by = [self.EmbeddingStore.collection_id == collection.uuid]

            if filter:
                filter_clauses = self._create_filter_clause(filter)
                if filter_clauses is not None:
                    filter_by.append(filter_clauses)

            stmt = (
                select(self.EmbeddingStore)
                .filter(*filter_by)
                .order_by(self.EmbeddingStore.document.op("@@")(query).desc())
                .limit(k)
            )

            results = session.execute(stmt).scalars().all()

            return [
                Document(
                    id=result.id,
                    page_content=result.document,
                    metadata=result.cmetadata or {},
                )
                for result in results
            ]

    def keyword_search_with_score(
        self,
        query: str,
        k: int = 10,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        """
        Keyword search with relevance score
        """
        with self._make_sync_session() as session:
            collection = self.get_collection(session)
            if not collection:
                raise ValueError("Collection not found")

            # Build filter conditions
            filter_conditions = ["collection_id = :collection_id"]
            params = {"collection_id": collection.uuid, "query": query, "k": k}

            if filter:
                # Handle metadata filtering
                for key, value in filter.items():
                    filter_conditions.append(f"cmetadata->>'{key}' = :filter_{key}")
                    params[f"filter_{key}"] = str(value)

            # Use pgroonga_score(tableoid, ctid) recommended approach
            stmt = text(
                f"""
                SELECT id, document, cmetadata, 
                       pgroonga_score(tableoid, ctid) as score
                FROM langchain_pg_embedding
                WHERE {' AND '.join(filter_conditions)}
                  AND document @@ :query
                ORDER BY pgroonga_score(tableoid, ctid) DESC
                LIMIT :k
            """
            )

            results = session.execute(stmt, params).all()

            return [
                (
                    Document(
                        id=result.id,
                        page_content=result.document,
                        metadata=result.cmetadata or {},
                    ),
                    float(result.score),
                )
                for result in results
            ]

    async def akeyword_search_with_score(
        self,
        query: str,
        k: int = 10,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        """Async version of keyword_search_with_score"""
        with self._make_sync_session() as session:
            collection = self.get_collection(session)
            if not collection:
                raise ValueError("Collection not found")

            filter_conditions = ["collection_id = :collection_id"]
            params = {"collection_id": collection.uuid, "query": query, "k": k}

            if filter:
                for key, value in filter.items():
                    filter_conditions.append(f"cmetadata->>'{key}' = :filter_{key}")
                    params[f"filter_{key}"] = str(value)

            stmt = text(
                f"""
                SELECT id, document, cmetadata, 
                       pgroonga_score(tableoid, ctid) as score
                FROM langchain_pg_embedding
                WHERE {' AND '.join(filter_conditions)}
                  AND document @@ :query
                ORDER BY pgroonga_score(tableoid, ctid) DESC
                LIMIT :k
            """
            )

            results = session.execute(stmt, params).all()

            return [
                (
                    Document(
                        id=result.id,
                        page_content=result.document,
                        metadata=result.cmetadata or {},
                    ),
                    float(result.score),
                )
                for result in results
            ]

    # ==================== Hybrid Search ====================

    def hybrid_search(
        self,
        query: str,
        k: int = 10,
        top_k_vector: int = 20,
        top_k_keyword: int = 20,
        alpha: float = 0.5,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        """
        Hybrid search: Combine vector search and keyword search

        Args:
            query: Search query
            k: Final result count
            top_k_vector: Top k results from vector search
            top_k_keyword: Top k results from keyword search
            alpha: Fusion weight (0=keyword only, 1=vector only)
            filter: Optional metadata filter

        Returns:
            List[Tuple[Document, float]]: Documents and fusion scores
        """
        # Get vector search results
        vector_results = self.similarity_search_with_score(
            query=query, k=top_k_vector, filter=filter
        )

        # Get keyword search results
        keyword_results = self.keyword_search_with_score(
            query=query, k=top_k_keyword, filter=filter
        )

        # Fuse using RRF (Reciprocal Rank Fusion)
        return self._rrf_fusion(
            vector_results=vector_results,
            keyword_results=keyword_results,
            k=k,
            alpha=alpha,
        )

    async def ahybrid_search(
        self,
        query: str,
        k: int = 10,
        top_k_vector: int = 20,
        top_k_keyword: int = 20,
        alpha: float = 0.5,
        filter: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        """Async version of hybrid_search"""
        # Get vector search results (async)
        vector_results = await self.asimilarity_search_with_score(
            query=query, k=top_k_vector, filter=filter
        )

        # Get keyword search results (async)
        keyword_results = await self.akeyword_search_with_score(
            query=query, k=top_k_keyword, filter=filter
        )

        # Fuse using RRF
        return self._rrf_fusion(
            vector_results=vector_results,
            keyword_results=keyword_results,
            k=k,
            alpha=alpha,
        )

    def _rrf_fusion(
        self,
        vector_results: List[Tuple[Document, float]],
        keyword_results: List[Tuple[Document, float]],
        k: int,
        alpha: float = 0.5,
        rrf_k: int = 60,
    ) -> List[Tuple[Document, float]]:
        """
        RRF (Reciprocal Rank Fusion) fusion algorithm

        Args:
            vector_results: Vector search results
            keyword_results: Keyword search results
            k: Final result count
            alpha: Vector result weight
            rrf_k: RRF constant

        Returns:
            List[Tuple[Document, float]]: Fused results
        """
        scored_docs = {}

        # Process vector search results
        for rank, (doc, score) in enumerate(vector_results):
            doc_id = doc.id
            if doc_id not in scored_docs:
                scored_docs[doc_id] = {
                    "doc": doc,
                    "rrf_score": 0,
                    "vector_score": 0,
                    "keyword_score": 0,
                }
            # RRF score
            scored_docs[doc_id]["rrf_score"] += alpha / (rrf_k + rank + 1)
            scored_docs[doc_id]["vector_score"] = score

        # Process keyword search results
        for rank, (doc, score) in enumerate(keyword_results):
            doc_id = doc.id
            if doc_id not in scored_docs:
                scored_docs[doc_id] = {
                    "doc": doc,
                    "rrf_score": 0,
                    "vector_score": 0,
                    "keyword_score": 0,
                }
            # RRF score
            scored_docs[doc_id]["rrf_score"] += (1 - alpha) / (rrf_k + rank + 1)
            scored_docs[doc_id]["keyword_score"] = score

        # Sort by RRF score
        sorted_results = sorted(
            scored_docs.values(), key=lambda x: x["rrf_score"], reverse=True
        )[:k]

        return [(item["doc"], item["rrf_score"]) for item in sorted_results]

    # ==================== Override Initialization ====================

    def __post_init__(self) -> None:
        """Override initialization to automatically create indexes"""
        super().__post_init__()
        self.create_search_indexes()

    async def __apost_init__(self) -> None:
        """Override async initialization to automatically create indexes"""
        await super().__apost_init__()
        await self.acreate_search_indexes()

In [ ]:
vector_store = HybridRAGStore(
    connection=POSTGRESQL_DATABASE_URL,
    embeddings=embeddings,
    collection_name="my_rag_docs",
)

In [ ]:
docs = [
    Document(
        page_content="there are cats in the pond",
        metadata={"id": 1, "location": "pond", "topic": "animals"},
    ),
    Document(
        page_content="ducks are also found in the pond",
        metadata={"id": 2, "location": "pond", "topic": "animals"},
    ),
    Document(
        page_content="fresh apples are available at the market",
        metadata={"id": 3, "location": "market", "topic": "food"},
    ),
    Document(
        page_content="the market also sells fresh oranges",
        metadata={"id": 4, "location": "market", "topic": "food"},
    ),
    Document(
        page_content="the new art exhibit is fascinating",
        metadata={"id": 5, "location": "museum", "topic": "art"},
    ),
    Document(
        page_content="a sculpture exhibit is also at the museum",
        metadata={"id": 6, "location": "museum", "topic": "art"},
    ),
    Document(
        page_content="a new coffee shop opened on Main Street",
        metadata={"id": 7, "location": "Main Street", "topic": "food"},
    ),
    Document(
        page_content="the book club meets at the library",
        metadata={"id": 8, "location": "library", "topic": "reading"},
    ),
    Document(
        page_content="the library hosts a weekly story time for kids",
        metadata={"id": 9, "location": "library", "topic": "reading"},
    ),
    Document(
        page_content="a cooking class for beginners is offered at the community center",
        metadata={"id": 10, "location": "community center", "topic": "classes"},
    ),
]

vector_store.add_documents(docs, ids=[doc.metadata["id"] for doc in docs])

In [ ]:
vector_store.hybrid_search(query="cat")